# Sesión 3: RAG y evaluación

En esta sesión se implementa la parte generativa del sistema RAG. El archivo `retrieval_results.csv` contiene los candidatos recuperados por FAISS para cada usuario, y el LLM se utiliza para reordenarlos y justificar las recomendaciones finales.

## 2. Imports

In [1]:
import pandas as pd
import numpy as np
import json
import re
import os
import ast

try:
    import ollama
except ImportError:
    ollama = None
    print("No se ha encontrado el paquete 'ollama'. Instálalo en el entorno si quieres ejecutar el LLM local.")

from tqdm.auto import tqdm

c:\Users\Willy\Desktop\Master\Sistemas de Recuperación de Información y de Recomendación\Practica\Rag\rag-steam-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Rutas

In [2]:
GAMES_DATASET_PATH = "../outputs/games_dataset.csv"
USER_PROFILES_PATH = "../outputs/user_profiles.csv"
RETRIEVAL_RESULTS_PATH = "../outputs/retrieval_results.csv"
OUTPUTS_DIR = "../outputs"

os.makedirs(OUTPUTS_DIR, exist_ok=True)

## 4. Cargar datos

In [3]:
games_df = pd.read_csv(GAMES_DATASET_PATH)
user_profiles_df = pd.read_csv(USER_PROFILES_PATH)
retrieval_results = pd.read_csv(RETRIEVAL_RESULTS_PATH)

print("games_df")
display(games_df.head())

print("user_profiles_df")
display(user_profiles_df.head())

print("retrieval_results")
display(retrieval_results.head())

games_df


,item_id,item_name,num_reviews,document
0,251570,7 Days to Die,20,7 days to die this game is awesome its like mi...
1,250420,8BitMMO,20,8bitmmo this game is so much fun i love buildi...
2,113400,APB Reloaded,20,apb reloaded i do hate the fact that i lagged ...
3,346110,ARK: Survival Evolved,20,ark survival evolved so i play ark on and off ...
4,224540,Ace of Spades,20,ace of spades meh awesome great game yeah a pr...


user_profiles_df


,user_id,num_items,profile_text,consumed_item_ids,consumed_item_names
0,76561197970982479,15,counter strike global offensive rising storm r...,"[""730"", ""35450"", ""8930"", ""1250"", ""232090"", ""24...","[""Counter-Strike: Global Offensive"", ""Rising S..."
1,js41637,15,the elder scrolls v skyrim terraria saints row...,"[""72850"", ""105600"", ""55230"", ""620"", ""238010"", ...","[""The Elder Scrolls V: Skyrim"", ""Terraria"", ""S..."
2,evcentric,15,gnomoria fallout new vegas borderlands fallout...,"[""224500"", ""22380"", ""8980"", ""377160"", ""49520"",...","[""Gnomoria"", ""Fallout: New Vegas"", ""Borderland..."
3,Riot-Punch,15,grand theft auto iv street fighter iv the witc...,"[""12210"", ""21660"", ""292030"", ""72850"", ""12750"",...","[""Grand Theft Auto IV"", ""Street Fighter IV"", ""..."
4,doctr,15,counter strike global offensive borderlands 2 ...,"[""730"", ""49520"", ""311210"", ""550"", ""218620"", ""2...","[""Counter-Strike: Global Offensive"", ""Borderla..."


retrieval_results


,user_id,rank,item_id,item_name,score,profile_text,consumed_item_names
0,76561197970982479,1,42700,Call of Duty: Black Ops,0.697285,counter strike global offensive rising storm r...,"[""Counter-Strike: Global Offensive"", ""Rising S..."
1,76561197970982479,2,10090,Call of Duty: World at War,0.692121,counter strike global offensive rising storm r...,"[""Counter-Strike: Global Offensive"", ""Rising S..."
2,76561197970982479,3,33930,Arma 2: Operation Arrowhead,0.685877,counter strike global offensive rising storm r...,"[""Counter-Strike: Global Offensive"", ""Rising S..."
3,76561197970982479,4,10,Counter-Strike,0.672896,counter strike global offensive rising storm r...,"[""Counter-Strike: Global Offensive"", ""Rising S..."
4,76561197970982479,5,202970,Call of Duty: Black Ops II,0.669079,counter strike global offensive rising storm r...,"[""Counter-Strike: Global Offensive"", ""Rising S..."


## 5. Convertir columnas serializadas

In [4]:
def safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []


for column in ["consumed_item_ids", "consumed_item_names"]:
    if column in user_profiles_df.columns:
        user_profiles_df[column] = user_profiles_df[column].apply(safe_literal_eval)

user_profiles_df[["user_id", "consumed_item_ids", "consumed_item_names"]].head()

,user_id,consumed_item_ids,consumed_item_names
0,76561197970982479,"[730, 35450, 8930, 1250, 232090, 24960, 24980,...","[Counter-Strike: Global Offensive, Rising Stor..."
1,js41637,"[72850, 105600, 55230, 620, 238010, 28050, 242...","[The Elder Scrolls V: Skyrim, Terraria, Saints..."
2,evcentric,"[224500, 22380, 8980, 377160, 49520, 28050, 10...","[Gnomoria, Fallout: New Vegas, Borderlands, Fa..."
3,Riot-Punch,"[12210, 21660, 292030, 72850, 12750, 8870, 620...","[Grand Theft Auto IV, Street Fighter IV, The W..."
4,doctr,"[730, 49520, 311210, 550, 218620, 22380, 37716...","[Counter-Strike: Global Offensive, Borderlands..."


## 6. Seleccionar usuarios para la demo

In [5]:
selected_users = user_profiles_df["user_id"].head(5).tolist()
selected_users

['76561197970982479', 'js41637', 'evcentric', 'Riot-Punch', 'doctr']

## 7. Obtener candidatos de usuario

In [6]:
def get_user_candidates(user_id, retrieval_results, top_n=10):
    user_results = retrieval_results[retrieval_results["user_id"].astype(str) == str(user_id)].copy()
    user_results["rank"] = pd.to_numeric(user_results["rank"], errors="coerce")
    user_results = user_results.sort_values("rank", ascending=True)

    relevant_columns = ["rank", "item_id", "item_name", "score"]
    return user_results[relevant_columns].head(top_n).reset_index(drop=True)

## 8. Construir prompt RAG

In [7]:
def build_rag_prompt(user_profile, candidates_df):
    profile_text = str(user_profile.get("profile_text", ""))[:2500]

    candidate_lines = []
    for _, row in candidates_df.iterrows():
        candidate_lines.append(
            f"{int(row['rank'])}. FAISS rank: {int(row['rank'])} | Juego: {row['item_name']} | score: {row['score']:.4f}"
        )
    candidates_text = "\n".join(candidate_lines)

    prompt = f"""
Actúas como un sistema de recomendación de videojuegos basado en contenido.

Contexto del usuario:
{profile_text}

Candidatos recuperados por FAISS:
{candidates_text}

Instrucciones:
- Reordena los 3 mejores juegos según afinidad real con el perfil del usuario.
- Justifica cada recomendación en una frase breve.
- No inventes juegos.
- Solo puedes elegir juegos de la lista de candidatos.
- Si algún candidato es irrelevante, ignóralo.
- Devuelve exclusivamente JSON válido, sin texto antes ni después.

Formato exacto pedido:
[
  {{
    "ranking": 1,
    "titulo": "Nombre exacto del juego",
    "razonamiento": "Explicación breve"
  }}
]
""".strip()

    return prompt

## 9. Llamada a Ollama

In [8]:
def call_ollama(prompt, model_name="llama3.2"):
    if ollama is None:
        print("Ollama no está disponible en este entorno Python. Instala el paquete 'ollama' y ejecuta: ollama run llama3.2")
        return None

    try:
        response = ollama.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.0},
        )
        return response["message"]["content"].strip()
    except Exception as exc:
        print(f"No se pudo llamar a Ollama con el modelo '{model_name}': {exc}")
        print("Asegúrate de tener Ollama abierto y ejecuta en una terminal: ollama run llama3.2")
        return None

## 10. Extraer JSON de la respuesta

In [9]:
def extract_json_response(response_text):
    if not response_text:
        return []

    cleaned = response_text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, list) else []
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[\s*\{.*?\}\s*\]", cleaned, flags=re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            return parsed if isinstance(parsed, list) else []
        except json.JSONDecodeError:
            return []

    return []

## 11. Filtro de alucinaciones

In [10]:
def hallucination_filter(llm_items, candidates_df):
    candidate_titles = {str(title).strip() for title in candidates_df["item_name"].dropna().tolist()}
    valid_items = []
    hallucinated_items = []

    for position, item in enumerate(llm_items, start=1):
        if not isinstance(item, dict):
            hallucinated_items.append({"titulo": None, "item": item, "motivo": "Elemento JSON no válido"})
            continue

        title = str(item.get("titulo", "")).strip()
        if title in candidate_titles:
            valid_items.append(
                {
                    "ranking": int(item.get("ranking", position)),
                    "titulo": title,
                    "razonamiento": str(item.get("razonamiento", "")).strip(),
                }
            )
        else:
            hallucinated_items.append({"titulo": title, "item": item, "motivo": "Título fuera de candidatos"})

    return valid_items, hallucinated_items

## 12. Comparar ranking FAISS y ranking LLM

In [11]:
def compare_rankings(valid_items, candidates_df):
    candidates = candidates_df.copy()
    candidates["item_name_clean"] = candidates["item_name"].astype(str).str.strip()

    rows = []
    for position, item in enumerate(valid_items, start=1):
        title = str(item.get("titulo", "")).strip()
        candidate_match = candidates[candidates["item_name_clean"] == title]
        if candidate_match.empty:
            continue

        rank_faiss = int(candidate_match.iloc[0]["rank"])
        rank_llm = int(item.get("ranking", position))
        rows.append(
            {
                "titulo": title,
                "rank_faiss": rank_faiss,
                "rank_llm": rank_llm,
                "cambio_rank": rank_faiss - rank_llm,
            }
        )

    return pd.DataFrame(rows, columns=["titulo", "rank_faiss", "rank_llm", "cambio_rank"])

## 13. Bucle principal RAG

In [12]:
rag_outputs = []
ranking_comparisons = []
hallucinations = []

for user_id in tqdm(selected_users, desc="Procesando usuarios"):
    user_profile_rows = user_profiles_df[user_profiles_df["user_id"].astype(str) == str(user_id)]
    if user_profile_rows.empty:
        print(f"No se encontró perfil para user_id={user_id}")
        continue

    user_profile = user_profile_rows.iloc[0]
    candidates_df = get_user_candidates(user_id, retrieval_results, top_n=10)
    if candidates_df.empty:
        print(f"No se encontraron candidatos para user_id={user_id}")
        continue

    prompt = build_rag_prompt(user_profile, candidates_df)
    raw_response = call_ollama(prompt)
    parsed_response = extract_json_response(raw_response)
    valid_response, hallucinated_items = hallucination_filter(parsed_response, candidates_df)
    comparison_df = compare_rankings(valid_response, candidates_df)

    rag_outputs.append(
        {
            "user_id": str(user_id),
            "raw_response": raw_response,
            "parsed_response": parsed_response,
            "valid_response": valid_response,
            "num_hallucinations": len(hallucinated_items),
        }
    )

    for item in hallucinated_items:
        hallucinations.append(
            {
                "user_id": str(user_id),
                "titulo": item.get("titulo"),
                "motivo": item.get("motivo"),
                "item": item.get("item"),
            }
        )

    for row in comparison_df.to_dict(orient="records"):
        row["user_id"] = str(user_id)
        ranking_comparisons.append(row)

print(f"Usuarios procesados: {len(rag_outputs)}")

Procesando usuarios: 100%|██████████| 5/5 [00:11<00:00,  2.33s/it]

Usuarios procesados: 5


## 14. Guardar resultados

In [13]:
rag_outputs_path = os.path.join(OUTPUTS_DIR, "rag_outputs.json")
ranking_comparisons_path = os.path.join(OUTPUTS_DIR, "ranking_comparisons.csv")
hallucinations_path = os.path.join(OUTPUTS_DIR, "hallucinations.csv")

with open(rag_outputs_path, "w", encoding="utf-8") as f:
    json.dump(rag_outputs, f, ensure_ascii=False, indent=2)

ranking_comparisons_df = pd.DataFrame(
    ranking_comparisons,
    columns=["user_id", "titulo", "rank_faiss", "rank_llm", "cambio_rank"],
)
ranking_comparisons_df.to_csv(ranking_comparisons_path, index=False)

hallucinations_df = pd.DataFrame(
    hallucinations,
    columns=["user_id", "titulo", "motivo", "item"],
)
hallucinations_df.to_csv(hallucinations_path, index=False)

print("Resultados guardados en:")
print(rag_outputs_path)
print(ranking_comparisons_path)
print(hallucinations_path)

Resultados guardados en:
../outputs\rag_outputs.json
../outputs\ranking_comparisons.csv
../outputs\hallucinations.csv


## 15. Mostrar ejemplo

In [14]:
if rag_outputs:
    example_output = rag_outputs[0]
    example_user_id = example_output["user_id"]
    example_profile = user_profiles_df[user_profiles_df["user_id"].astype(str) == str(example_user_id)].iloc[0]
    example_candidates = get_user_candidates(example_user_id, retrieval_results, top_n=10)

    print("Perfil recortado:")
    print(str(example_profile["profile_text"])[:2500])

    print("\nCandidatos FAISS:")
    display(example_candidates)

    print("Respuesta válida del LLM:")
    display(pd.DataFrame(example_output["valid_response"]))

    print("Comparación de ranking:")
    display(ranking_comparisons_df[ranking_comparisons_df["user_id"].astype(str) == str(example_user_id)])
else:
    print("No hay salidas RAG para mostrar. Si Ollama no estaba disponible, ejecuta: ollama run llama3.2")

Perfil recortado:
counter strike global offensive rising storm red orchestra 2 multiplayer sid meier s civilization v killing floor killing floor 2 battlefield bad company 2 mass effect 2 day of defeat source dragon age origins plants vs zombies game of the year xcom enemy unknown just cause 2 borderlands insurgency final fantasy vii counter strike global offensive hold shift to win hold ctrl to lose zika do baile best game in the bloody world good game it was very fun i really do recommend anyone who has played any cs or cod like games to give this game a go it may not be like cod but it is still as much fun maybe even better keepitreal sees player tries shoot at the head fails dies vote kick player rage quitsalloha snackbar ค มก บท ซ อมาม นส มากคร บ great game would recommend best game evar 10 10 ign actually such an addictive game really entertaining and extremely competitive at times i d recommend it to you cause its so good thats really it it s a great tdm style game motto of the 

,rank,item_id,item_name,score
0,1,42700,Call of Duty: Black Ops,0.697285
1,2,10090,Call of Duty: World at War,0.692121
2,3,33930,Arma 2: Operation Arrowhead,0.685877
3,4,10,Counter-Strike,0.672896
4,5,202970,Call of Duty: Black Ops II,0.669079
5,6,298240,Red Crucible: Firestorm,0.661126
6,7,259700,Dead Sky,0.656392
7,8,437220,The Culling,0.656224
8,9,204450,Call of Juarez Gunslinger,0.655868
9,10,209670,Cortex Command,0.654048


Respuesta válida del LLM:


,ranking,titulo,razonamiento
0,1,Counter-Strike,Se ajusta a la afición del usuario por juegos ...
1,2,Call of Duty: Black Ops,Comparte elementos de estrategia y decisión co...


Comparación de ranking:


,user_id,titulo,rank_faiss,rank_llm,cambio_rank
0,76561197970982479,Counter-Strike,4,1,3
1,76561197970982479,Call of Duty: Black Ops,1,2,-1


## 16. Evaluación simple

In [15]:
num_users_processed = len(rag_outputs)
total_valid_recommendations = sum(len(output["valid_response"]) for output in rag_outputs)
total_hallucinations = sum(output["num_hallucinations"] for output in rag_outputs)
responses_without_hallucinations = sum(output["num_hallucinations"] == 0 for output in rag_outputs)

if num_users_processed > 0:
    pct_without_hallucinations = responses_without_hallucinations / num_users_processed * 100
else:
    pct_without_hallucinations = 0.0

if not ranking_comparisons_df.empty:
    num_rank_changes = int((ranking_comparisons_df["cambio_rank"] != 0).sum())
    mean_abs_rank_change = float(ranking_comparisons_df["cambio_rank"].abs().mean())
else:
    num_rank_changes = 0
    mean_abs_rank_change = 0.0

evaluation_summary = pd.DataFrame(
    [
        {
            "usuarios_procesados": num_users_processed,
            "recomendaciones_validas": total_valid_recommendations,
            "alucinaciones_totales": total_hallucinations,
            "porcentaje_respuestas_sin_alucinaciones": pct_without_hallucinations,
            "veces_que_el_llm_cambio_el_orden": num_rank_changes,
            "cambio_medio_absoluto_de_ranking": mean_abs_rank_change,
        }
    ]
)

display(evaluation_summary)

,usuarios_procesados,recomendaciones_validas,alucinaciones_totales,porcentaje_respuestas_sin_alucinaciones,veces_que_el_llm_cambio_el_orden,cambio_medio_absoluto_de_ranking
0,5,11,4,60.0,3,0.454545


## 17. Evaluación cualitativa

La salida del RAG se considera razonable cuando el LLM selecciona juegos presentes en los candidatos, mantiene coherencia con el perfil del usuario y proporciona justificaciones específicas. El filtro de alucinaciones permite comprobar que no se recomiendan juegos inexistentes o no recuperados. La comparación entre el ranking FAISS y el ranking del LLM permite observar si el modelo generativo actúa solo como explicador o si también modifica el orden de los candidatos.

## 18. Conclusión

Con esta sesión se completa el pipeline RAG de la práctica. FAISS actúa como recuperador de candidatos, mientras que el LLM actúa como re-ranker y generador de explicaciones breves para las recomendaciones. Además, se aplica un filtro de alucinaciones para verificar que las recomendaciones pertenecen a la lista recuperada y se generan los archivos finales necesarios para el informe.